# Train a Kamba voice (Understand Kamba)

Fine-tunes one of Meta's small MMS voices on recordings of **one Kikamba speaker**, so it speaks in her voice and
learns Kikamba pronunciation. The result runs fast on an ordinary **CPU**, so it never depends on a GPU allowance.

Pick the speaker and the starting voice in **step 0**:

| `DATASET` | Speaker | Amount |
|---|---|---|
| `lawmaluki/kamba-tts-ddd-speaker-d` | Speaker D from Digital Divide Data's public Kamba dataset (CC-BY-4.0), woman, 66 | 894 clips, 2.2 h (recommended) |
| `lawmaluki/kamba-tts-ddd-speaker-a` | Speaker A from the same dataset, woman, 47 | 216 clips, 28 min |
| `lawmaluki/kamba-voice-tts-contributor` | The contributor who recorded on /contribute | 3 minutes (experiment) |

`BASE = "kik"` starts from the **Kikuyu** voice (closest relative of Kikamba; knows ĩ and ũ). `"swh"` starts from Swahili.

**Before running (Kaggle):** right panel → *Session options*: **GPU T4 x2**, **Internet On**; menu **Add-ons → Secrets**:
`HF_TOKEN` (a Hugging Face token that can read your private datasets), attached to this notebook. Then **Run All**.
For a long run, use **Save Version → Save & Run All (Commit)** so it keeps going with the browser closed.

In [ ]:
# 0. Choices
DATASET = "lawmaluki/kamba-tts-ddd-speaker-d"
BASE = "kik"            # "kik" (Kikuyu, knows ĩ/ũ) or "swh" (Swahili)
TRAIN_MINUTES = 60      # rough training budget on a T4; the number of epochs is derived from it

In [ ]:
# 1. Setup (~3 min): check GPU, internet and token; install the voice-training tool
import os, socket, subprocess, torch
assert torch.cuda.is_available(), "No GPU: Session options -> Accelerator -> GPU T4 x2"
try:
    socket.create_connection(("huggingface.co", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet: Session options -> Internet -> On")

token = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not token:
    raise SystemExit("Add your Hugging Face token: Add-ons -> Secrets -> HF_TOKEN (and attach it to this notebook)")
# Passed as an environment variable: importing huggingface_hub here, before the pip install below
# replaces it with another version, breaks downloads later in the same session.
os.environ["HF_TOKEN"] = token

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.chdir(WORK)
if not os.path.isdir("finetune-hf-vits"):
    !git clone -q https://github.com/ylacombe/finetune-hf-vits.git
os.chdir(f"{WORK}/finetune-hf-vits")
!git checkout -q 6f3f51f4d667f5c3eef89484d151ffd39d2c2b89
# Versions this tool was tested with (it needs pieces newer transformers/matplotlib removed).
# Red "dependency resolver" messages about other Kaggle packages are harmless.
!pip install -q "transformers==4.45.2" "datasets[audio]==3.6.0" "accelerate==1.1.1" "matplotlib==3.9.4" "huggingface_hub==0.36.2" tensorboard Cython soundfile librosa
# Kaggle's newer hf_xet helper may not match that huggingface_hub; plain HTTP downloads are fine here.
!pip uninstall -y -q hf_xet
!cd monotonic_align && mkdir -p monotonic_align && python setup.py build_ext --inplace -q
# One-speaker data has no speaker ids; let the script cope with that.
!sed -i 's/speaker_id=batch\["speaker_id"\]/speaker_id=batch.get("speaker_id")/g' run_vits_finetuning.py
import huggingface_hub  # first import happens only now, after the install
assert huggingface_hub.__version__ == "0.36.2", f"unexpected huggingface_hub {huggingface_hub.__version__}: Run -> Restart & Run All"
print("Ready:", torch.cuda.get_device_name(0), "| huggingface_hub", huggingface_hub.__version__)

In [ ]:
# 2. Starting voice (~2 min): the MMS voice plus the discriminator needed to train it
if not os.path.isdir(f"mms-{BASE}-train"):
    !python convert_original_discriminator_checkpoint.py --language_code {BASE} --pytorch_dump_folder_path mms-{BASE}-train
print(sorted(os.listdir(f"mms-{BASE}-train")))

In [ ]:
# 3. Train. The dataset has the text written for each starting voice (text_kik keeps ĩ/ũ; text_swh writes i/u).
import json, math
from datasets import load_dataset
n_clips = load_dataset(DATASET, split="train").num_rows
batch = 16 if n_clips >= 200 else 8
steps_per_epoch = math.ceil(n_clips / batch)
epochs = max(30, min(500, int(TRAIN_MINUTES * 60 / 0.7 / steps_per_epoch)))  # ~0.7 s per step on a T4
print(f"{n_clips} clips -> {epochs} epochs, ~{epochs * steps_per_epoch} steps")
config = {
    "project_name": "kamba_voice",
    "push_to_hub": False,
    "report_to": ["tensorboard"],
    "overwrite_output_dir": True,
    "output_dir": f"{WORK}/kamba-voice-ft",
    "dataset_name": DATASET,
    "audio_column_name": "audio",
    "text_column_name": f"text_{BASE}",
    "train_split_name": "train",
    "eval_split_name": "train",
    "full_generation_sample_text": "wĩoo mũseo mũsyĩ waku ũilyĩ ata" if BASE == "kik" else "wioo museo musyi waku uilyi ata",
    "max_duration_in_seconds": 20,
    "min_duration_in_seconds": 1.0,
    "max_tokens_length": 500,
    "model_name_or_path": f"mms-{BASE}-train",
    "preprocessing_num_workers": 2,
    "do_train": True,
    "num_train_epochs": epochs,
    "gradient_accumulation_steps": 1,
    "per_device_train_batch_size": batch,
    "learning_rate": 2e-5,
    "adam_beta1": 0.8,
    "adam_beta2": 0.99,
    "warmup_ratio": 0.01,
    "do_eval": True,
    "eval_steps": 500,
    "per_device_eval_batch_size": 4,
    "max_eval_samples": 4,
    "do_step_schedule_per_epoch": True,
    "weight_disc": 3, "weight_fmaps": 1, "weight_gen": 1, "weight_kl": 1.5, "weight_duration": 1, "weight_mel": 35,
    "fp16": True,
    "seed": 456,
}
json.dump(config, open("kamba_config.json", "w"), indent=1)
# Runs the training as a separate process and shows a progress line every ~2 minutes; the full log is
# in train_log.txt. If training stops, its last lines are printed below.
import subprocess, time, re
log = open(f"{WORK}/train_log.txt", "w")
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1")
proc = subprocess.Popen(["accelerate", "launch", "--num_processes", "1", "--mixed_precision", "fp16",
                         "run_vits_finetuning.py", "kamba_config.json"], stdout=log, stderr=subprocess.STDOUT, env=env)
start = time.time()
while proc.poll() is None:
    time.sleep(120)
    tail = open(f"{WORK}/train_log.txt", encoding="utf-8", errors="replace").read()[-20000:]
    steps = re.findall(r"Steps:\s+(\d+)%\|[^|]*\|\s*(\d+)/(\d+)\s*\[([^\]<]*)<([^,\]]*)", tail)
    if steps:
        pct, done, total, spent, left = steps[-1]
        print(f"{time.strftime('%H:%M')}  training {done}/{total} steps ({pct}%), about {left.strip()} left", flush=True)
    else:
        print(f"{time.strftime('%H:%M')}  preparing data / loading the model ({int(time.time() - start) // 60} min)", flush=True)
log.close()
if proc.returncode != 0 or not os.path.exists(f"{WORK}/kamba-voice-ft/model.safetensors"):
    lines = open(f"{WORK}/train_log.txt", encoding="utf-8", errors="replace").read().splitlines()
    print(chr(10).join(l for l in lines[-40:] if "it/s]" not in l))
    raise SystemExit(f"Training stopped (exit code {proc.returncode}) - copy the lines above to Claude")
print("Trained voice saved to", f"{WORK}/kamba-voice-ft")

In [ ]:
# 4. Listen: the same sentences in the Swahili voice, the plain Kikuyu voice and the trained voice
import unicodedata, numpy as np, soundfile as sf
from IPython.display import Audio, display, HTML
from transformers import VitsModel, AutoTokenizer

def prep(text, base):  # text written the way each voice reads it (same rules as the training data)
    t = unicodedata.normalize("NFC", text.replace("’", "'").lower())
    t = t.replace("î", "ĩ").replace("û", "ũ").replace("ū", "ũ")       # other spellings of the tilde vowels
    keep = "ĩũ" if base == "kik" else ""
    t = "".join(c if c in keep else "".join(x for x in unicodedata.normalize("NFD", c) if not unicodedata.combining(x)) for c in t)
    vocab = set(" '-01245_abcdefghijklmnopqrstuvwyzĩũ") if base == "kik" else set(" '-012345678_abcdefghijklmnopqrstuvwyz")
    return " ".join("".join(c if c in vocab else " " for c in t).split())

voices = {"Swahili voice (app fallback)": ("facebook/mms-tts-swh", "swh"),
          "Kikuyu voice (untrained)": ("facebook/mms-tts-kik", "kik"),
          "trained Kamba voice": (f"{WORK}/kamba-voice-ft", BASE)}
loaded = {name: (VitsModel.from_pretrained(p).to("cuda").eval(), AutoTokenizer.from_pretrained(p), b) for name, (p, b) in voices.items()}
def speak(name, text):
    m, t, b = loaded[name]
    torch.manual_seed(0)
    with torch.inference_mode():
        return m(**t(prep(text, b), return_tensors="pt").to("cuda")).waveform[0].float().cpu().numpy()

listen = ["Wĩoo mũseo, mũsyĩ waku ũilyĩ ata?", "Nĩ mũvea mũno nũndũ wa ũtethyo waku.",
          "Ũmũnthĩ nĩkwĩthĩwa na mbua mbingĩ kĩsionĩ kya Machakos.", "Twĩ na mũtũkĩ wa kũthi ũthũkũmonĩ kĩla kwakya."]
os.makedirs(f"{WORK}/samples", exist_ok=True)
for i, s in enumerate(listen, 1):
    display(HTML(f"<b>{i}. {s}</b>"))
    for name in voices:
        w = speak(name, s)
        sf.write(f"{WORK}/samples/{i}-{name.split()[0].lower()}.wav", w, 16000)
        display(HTML(name)); display(Audio(w, rate=16000))
print("WAV files also saved in", f"{WORK}/samples (Output tab -> download)")

In [ ]:
# 5. Clarity test: 20 Kikamba sentences the speaker did NOT record, read by each voice and transcribed by the
#    app's Kikamba speech recogniser. Lower character error = clearer. (A proxy, not a listening test.)
import urllib.request, csv, io
recorded = set(load_dataset(DATASET, split="train")["text_original"])
tsv = urllib.request.urlopen("https://raw.githubusercontent.com/lawmaluki/understand-kamba/main/app/data/kikamba_prompts.tsv").read().decode("utf-8")
prompts = [r["text"] for r in csv.DictReader((l for l in io.StringIO(tsv) if not l.startswith("#")), delimiter="\t")]
test = [p for p in prompts if p not in recorded and 40 <= len(p) <= 120][:20]

from transformers import AutoModelForCTC, AutoProcessor
asr_id = "lawmaluki/w2v-bert-2.0_kamba"
proc = AutoProcessor.from_pretrained(asr_id); asr = AutoModelForCTC.from_pretrained(asr_id).to("cuda").eval()
def transcribe(w):
    x = proc(w, sampling_rate=16000, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        return proc.decode(asr(**x).logits.argmax(-1)[0])
def edits(a, b):
    d = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        p, d[0] = d[0], i
        for j, cb in enumerate(b, 1):
            p, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, p + (ca != cb))
    return d[-1]
for name in voices:
    err = tot = 0
    for s in test:
        ref, hyp = prep(s, "swh"), prep(transcribe(speak(name, s)), "swh")  # compare without tildes (the recogniser writes none)
        err += edits(ref, hyp); tot += len(ref)
    print(f"{name:30} character error {100 * err / tot:5.1f}%  (on {len(test)} unseen sentences)")
print("Lower is better.")

## 6. (Optional) Save the trained voice to your Hugging Face account

Only if it sounds good and step 5 shows it is clearer than the Swahili voice. Set `UPLOAD = True` and run.
Saved **privately**; the app can then use it as its always-available CPU voice. If you trained on Digital Divide
Data's recordings, credit them (CC-BY-4.0) wherever the voice is published.

In [ ]:
UPLOAD = False
if UPLOAD:
    from huggingface_hub import HfApi
    repo = "lawmaluki/mms-tts-kam-" + DATASET.split("/")[-1].replace("kamba-tts-", "").replace("kamba-voice-tts-", "")
    HfApi().create_repo(repo, private=True, exist_ok=True)
    HfApi().upload_folder(folder_path=f"{WORK}/kamba-voice-ft", repo_id=repo, ignore_patterns=["runs/*", "checkpoint*"])
    print("uploaded: https://huggingface.co/" + repo)